# Ollama Re-ranking Demo


In [8]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma

In [9]:
with open('docs/company_policy.txt', encoding='utf-8') as f:
    text=f.read()

documents=[Document(page_content=text)]

In [10]:
splitter=RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks=splitter.split_documents(documents)

In [11]:
embeddings=OllamaEmbeddings(model='nomic-embed-text')
vectorstore=Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory='./chroma_db')
retriever=vectorstore.as_retriever(search_kwargs={'k':5})

In [12]:
llm=ChatOllama(model='gpt-oss:120b-cloud', temperature=0)

In [13]:
def score_document(question, document):
    prompt=f'''Question:\n{question}\n\nDocument:\n{document}\n\nReturn ONLY a relevance score from 0 to 100.'''
    response=llm.invoke(prompt)
    try:
        return float(response.content.strip())
    except:
        return 0.0

In [14]:
question='What is maternity leave policy?'
retrieved_docs=retriever.invoke(question)
scored=[]
for doc in retrieved_docs:
    score=score_document(question, doc.page_content)
    scored.append((doc, score))
    print(score, doc.page_content)

top_docs=sorted(scored,key=lambda x:x[1],reverse=True)[:2]

100.0 Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.
100.0 Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.


In [15]:
context='\n\n'.join(doc.page_content for doc,score in top_docs)
response=llm.invoke(f'Context:\n{context}\n\nQuestion:{question}')
print(response.content)

**Maternity Leave Policy**

- **Eligibility:** All employees qualify for maternity leave.
- **Duration:** Up to **6 months** of paid (or as stipulated by the company’s compensation structure) leave is provided.

That’s the entirety of the maternity‑leave provision given in the provided policy details.
